# Comparação justa: modelo linear × MLP (8 espécies)

Na Etapa E, o modelo linear de referência foi treinado com SGD (passo 0,1, 2000 épocas), sem ajuste
de hiperparâmetros, enquanto a MLP teve 22 configurações testadas. Para que a diferença entre os
dois reflita a camada oculta, e não o treino, o linear recebe aqui o mesmo tratamento:

- mesma função custo (entropia cruzada com peso por classe), mesmo mini-batch (64), mesmas sementes;
- algumas configurações de otimizador e passo, escolhidas **só pela validação** (acurácia balanceada
  média de 3 sementes), como na Etapa D;
- a configuração escolhida é avaliada **uma única vez** no teste.

A MLP final não é alterada.

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

dados = np.load("../dados_processados/etapa2.npz", allow_pickle=True)
X = dados["X_simples"].astype(np.float32)
ESPECIES = pd.Series(dados["y_simples"]).value_counts().index.tolist()
N = len(ESPECIES)
y = np.array([ESPECIES.index(e) for e in dados["y_simples"]])
pt, pv, pte = dados["pos_treino_simples"], dados["pos_val_simples"], dados["pos_teste_simples"]
Xtr, ytr, Xva, yva, Xte, yte = X[pt], y[pt], X[pv], y[pv], X[pte], y[pte]
PESOS = len(ytr) / (N * np.bincount(ytr, minlength=N))

In [2]:
def treina_linear(cfg, semente):
    torch.manual_seed(semente)
    modelo = nn.Sequential(nn.Linear(5, N))  # regressão softmax: sem camada oculta
    if cfg["otimizador"] == "adam":
        otim = torch.optim.Adam(modelo.parameters(), lr=cfg["lr"])
    elif cfg["otimizador"] == "sgd":
        otim = torch.optim.SGD(modelo.parameters(), lr=cfg["lr"])
    else:
        otim = torch.optim.SGD(modelo.parameters(), lr=cfg["lr"], momentum=0.9)
    torch.manual_seed(semente)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(PESOS, dtype=torch.float32))
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)), batch_size=64, shuffle=True)
    melhor, melhores, melhor_ep = float("inf"), None, -1
    for ep in range(cfg["epocas"]):
        modelo.train()
        for xb, yb in loader:
            otim.zero_grad()
            crit(modelo(xb), yb).backward()
            otim.step()
        modelo.eval()
        with torch.no_grad():
            pval = crit(modelo(torch.from_numpy(Xva)), torch.from_numpy(yva)).item()
        if pval < melhor:
            melhor, melhores, melhor_ep = pval, copy.deepcopy(modelo.state_dict()), ep + 1
    modelo.load_state_dict(melhores)
    modelo.eval()
    return modelo, melhor_ep


def acuracia_balanceada(modelo, Xc, yc):
    with torch.no_grad():
        p = modelo(torch.from_numpy(Xc)).argmax(dim=1).numpy()
    return np.mean([np.mean(p[yc == c] == c) for c in range(N)]), p

## Escolha na validação

Mesma regra da Etapa D: fica a configuração de maior acurácia balanceada média; o treino original da
Etapa E (SGD 0,1, 2000 épocas) é o ponto de partida.

In [3]:
CONFIGS = [
    {"nome": "SGD 0,1, 2000 épocas (Etapa E)", "otimizador": "sgd", "lr": 0.1, "epocas": 2000},
    {"nome": "SGD 0,1, 4000 épocas", "otimizador": "sgd", "lr": 0.1, "epocas": 4000},
    {"nome": "Adam 0,01, 1000 épocas (como a MLP)", "otimizador": "adam", "lr": 0.01, "epocas": 1000},
    {"nome": "Adam 0,001, 1000 épocas", "otimizador": "adam", "lr": 0.001, "epocas": 1000},
    {"nome": "SGD com momento 0,01, 1000 épocas", "otimizador": "momento", "lr": 0.01, "epocas": 1000},
]
linhas, modelos = [], {}
for cfg in CONFIGS:
    abs_, eps = [], []
    for s in [0, 1, 2]:
        m, ep = treina_linear(cfg, s)
        modelos[(cfg["nome"], s)] = m
        abs_.append(acuracia_balanceada(m, Xva, yva)[0])
        eps.append(ep)
    linhas.append({"configuração": cfg["nome"], "acurácia balanceada (val)": np.mean(abs_),
                   "desvio": np.std(abs_), "melhores épocas": eps})
    print(f"{cfg['nome']}: {np.mean(abs_):.3f} ± {np.std(abs_):.3f} (épocas {eps})")
selecao = pd.DataFrame(linhas)
escolhida = selecao.loc[selecao["acurácia balanceada (val)"].idxmax(), "configuração"]
print("\nEscolhida pela validação:", escolhida)
selecao.round(3)

SGD 0,1, 2000 épocas (Etapa E): 0.947 ± 0.002 (épocas [999, 1040, 1803])


SGD 0,1, 4000 épocas: 0.950 ± 0.001 (épocas [2949, 3136, 2583])


Adam 0,01, 1000 épocas (como a MLP): 0.953 ± 0.001 (épocas [581, 565, 681])


Adam 0,001, 1000 épocas: 0.943 ± 0.000 (épocas [1000, 990, 999])


SGD com momento 0,01, 1000 épocas: 0.946 ± 0.002 (épocas [1000, 947, 794])

Escolhida pela validação: Adam 0,01, 1000 épocas (como a MLP)


,configuração,acurácia balanceada (val),desvio,melhores épocas
0,"SGD 0,1, 2000 épocas (Etapa E)",0.947,0.002,"[999, 1040, 1803]"
1,"SGD 0,1, 4000 épocas",0.950,0.001,"[2949, 3136, 2583]"
2,"Adam 0,01, 1000 épocas (como a MLP)",0.953,0.001,"[581, 565, 681]"
3,"Adam 0,001, 1000 épocas",0.943,0.000,"[1000, 990, 999]"
4,"SGD com momento 0,01, 1000 épocas",0.946,0.002,"[1000, 947, 794]"


## Avaliação única no teste (configuração escolhida)

Comparação com a MLP final da Etapa E (acurácia balanceada no teste 0,959 ± 0,0002).

In [4]:
res = []
for s in [0, 1, 2]:
    ab, p = acuracia_balanceada(modelos[(escolhida, s)], Xte, yte)
    sens = [np.mean(p[yte == c] == c) for c in range(N)]
    prec = [np.mean(yte[p == c] == c) if (p == c).any() else 0.0 for c in range(N)]
    f1 = [2 * a * b / (a + b) if a + b else 0.0 for a, b in zip(prec, sens)]
    res.append({"semente": s, "acurácia balanceada": ab, "F1 macro": np.mean(f1), "acurácia": np.mean(p == yte)})
teste = pd.DataFrame(res)
print(f"Linear ({escolhida}) no teste:")
for k in ["acurácia balanceada", "F1 macro", "acurácia"]:
    print(f"  {k}: {teste[k].mean():.3f} ± {teste[k].std(ddof=0):.4f}")
print("MLP final no teste (Etapa E): acurácia balanceada 0,959; F1 macro 0,926; acurácia 0,955")
erro_lin, erro_mlp = 1 - teste["acurácia balanceada"].mean(), 1 - 0.9594
print(f"Erro em acurácia balanceada: linear {erro_lin:.3f} × MLP {erro_mlp:.3f} "
      f"({100 * (1 - erro_mlp / erro_lin):.0f}% menor na MLP)")

Linear (Adam 0,01, 1000 épocas (como a MLP)) no teste:
  acurácia balanceada: 0.942 ± 0.0010
  F1 macro: 0.901 ± 0.0015
  acurácia: 0.937 ± 0.0011
MLP final no teste (Etapa E): acurácia balanceada 0,959; F1 macro 0,926; acurácia 0,955
Erro em acurácia balanceada: linear 0.058 × MLP 0.041 (30% menor na MLP)
